# Fine-Tuning Interview Question & Follow-up Model (PreMock AI)
This notebook trains an open-source LLM (Llama-3-8B-Instruct or Mistral-7B) on your **10,000 interview examples dataset** for:
1. **Resume-to-Question Generation**
2. **Adaptive Follow-up Question Generation**
3. **Resume Knowledge Extraction**

⚡ **Runs on Google Colab's Free T4 GPU using Unsloth (fast QLoRA).**

In [ ]:
# 1. Install Unsloth and modern training dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets transformers triton

In [ ]:
# 2. Load 4-bit Llama-3-8B-Instruct Base Model
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detection (Float16 for T4)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = max_seq_length,
)
print("Model and LoRA adapters initialized successfully!")

In [ ]:
# 3. Load your uploaded training dataset (train_all.jsonl or train_questions.jsonl)
# In Colab, upload train_all.jsonl to the files panel on the left.
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = load_dataset("json", data_files="train_all.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"Loaded {len(dataset)} examples!")

In [ ]:
# 4. Train with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 120, # Adjust steps: 100-300 for quick testing, or num_train_epochs=1 for full
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()
print("Training complete!")

In [ ]:
# 5. Test Inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are an expert technical interviewer and interview design system. Task: role_question. Role: Full Stack Developer. Difficulty: intermediate. Always respond with a valid JSON object strictly matching the target schema."},
    {"role": "user", "content": "Instruction: Generate one interview question for the supplied role and resume profile.\n\nInput:\n{\"role\": {\"title\": \"Full Stack Developer\", \"skills\": [\"Next.js\", \"FastAPI\", \"PostgreSQL\"]}}"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 256, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# 6. Save LoRA Adapters or Export to GGUF (for Ollama)
model.save_pretrained("interview_lora_model")
tokenizer.save_pretrained("interview_lora_model")
print("Model saved to interview_lora_model/")

# Optional: Export directly to 16-bit or GGUF for local Ollama inference
# model.save_pretrained_gguf("interview_model_gguf", tokenizer, quantization_method = "q4_k_m")